# Summit.OS — Visual Detector Training (Colab)

Fine-tune **YOLOv8n** on the Summit.OS multi-domain dataset and export to ONNX.

Covers 18 operational classes across:
Wildfire, Search & Rescue, Oil/Hazmat, Agriculture, Maritime, Infrastructure, Wildlife.

**Outputs**
- `models/summit_detector.onnx` — ONNX detection model
- `models/summit_detector_classes.json` — class index map

Run all cells top-to-bottom (**Runtime → Run all**).
The final cells download both output files to your browser.

## Step 1 — GPU Runtime

> **Before running:** switch to a GPU runtime for significantly faster training.
>
> `Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save`

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU detected — training will run on CPU (very slow for YOLOv8).")

## Step 2 — Install Dependencies

In [ ]:
# Install required packages
%pip install -q ultralytics onnx onnxruntime
print("Dependencies installed.")

## Step 3 — Write Source Files to /content/

In [ ]:
%%writefile /content/onnx_compat.py
"""
Compatibility shim for skl2onnx on Python 3.14+ with onnx >= 1.19.

Python 3.14 stricter protobuf enforcement rejects numpy integer subclasses
(np.uint32, np.int64, np.bool_, etc.) and plain Python booleans in onnx
AttributeProto integer fields:

  TypeError: Field onnx.AttributeProto.ints: Expected an int, got a boolean.

Root cause: skl2onnx's random_forest / gradient_boosting converters pass
numpy-typed arrays for tree node attributes.

Patching onnx.helper.* alone is insufficient because skl2onnx._container
does `from onnx.helper import make_node, make_attribute` — a local binding
that bypasses module-level patches applied afterward. We therefore:

  1. Patch onnx.helper.make_attribute and make_node so any fresh import gets
     the shim automatically.
  2. Retroactively patch skl2onnx.common._container's local bindings if that
     module was already imported (e.g. when skl2onnx is imported before us).

Import this module BEFORE importing skl2onnx in any training script:
    import onnx_compat  # noqa: F401 — must be first
"""
import sys


def _coerce_ints(value):
    """Coerce bool/numpy-int scalar or iterable values to plain Python int."""
    if isinstance(value, (str, bytes)):
        return value
    # scalar bool
    if isinstance(value, bool):
        return int(value)
    # iterable (list, tuple, ndarray, generator, …)
    if hasattr(value, "__iter__"):
        try:
            coerced = []
            for v in value:
                if isinstance(v, bool):
                    coerced.append(int(v))
                elif type(v) is not int and hasattr(v, "__index__"):
                    # numpy int types: np.uint32, np.int64, np.intp, etc.
                    try:
                        coerced.append(int(v))
                    except Exception:
                        coerced.append(v)
                else:
                    coerced.append(v)
            return coerced
        except Exception:
            return value
    return value


try:
    import onnx.helper as _h

    _orig_make_attribute = _h.make_attribute
    _orig_make_node = _h.make_node

    def _patched_make_attribute(key, value, doc_string=None, attr_type=None):
        value = _coerce_ints(value)
        kwargs = {}
        if doc_string is not None:
            kwargs["doc_string"] = doc_string
        if attr_type is not None:
            kwargs["attr_type"] = attr_type
        return _orig_make_attribute(key, value, **kwargs)

    def _patched_make_node(op_type, inputs, outputs, name=None, doc_string=None,
                           domain=None, overload=None, **kwargs):
        clean_kwargs = {k: _coerce_ints(v) for k, v in kwargs.items()}
        extra = {}
        if name is not None:
            extra["name"] = name
        if doc_string is not None:
            extra["doc_string"] = doc_string
        if domain is not None:
            extra["domain"] = domain
        if overload is not None:
            extra["overload"] = overload
        return _orig_make_node(op_type, inputs, outputs, **extra, **clean_kwargs)

    # Patch the onnx.helper module so fresh imports get shims
    _h.make_attribute = _patched_make_attribute
    _h.make_node = _patched_make_node

    # Retroactively patch skl2onnx._container if it's already been imported
    # (it uses `from onnx.helper import make_node, make_attribute` — local bindings)
    _skl_container_name = "skl2onnx.common._container"
    if _skl_container_name in sys.modules:
        _c = sys.modules[_skl_container_name]
        if getattr(_c, "make_attribute", None) is _orig_make_attribute:
            _c.make_attribute = _patched_make_attribute
        if getattr(_c, "make_node", None) is _orig_make_node:
            _c.make_node = _patched_make_node

except ImportError:
    pass  # onnx not installed yet


In [ ]:
%%writefile /content/download_visual_datasets.py
"""
Download and prepare labeled image datasets for Summit.OS visual object detection training.

Covers all operational domains:
  - Wildfire / fire / smoke      — D-Fire dataset (aerial imagery)
  - Search & rescue              — SeaDronesSee (person-in-water, UAV) + AIDER (emergency response)
  - Oil pipeline / hazmat        — Synthetic generation (no suitable public dataset)
  - Agricultural                 — Synthetic generation
  - Maritime                     — COCO 2017 val subset (boats + persons)
  - Infrastructure               — Synthetic generation
  - Wildlife / dangerous animals — COCO 2017 val subset (bear, elephant, horse, etc.)

Output directory tree:
  packages/ml/data/
    images/fire_smoke/         ← D-Fire extracted images
    images/coco_subset/        ← COCO val images + filtered annotations
    images/seadronessee/       ← SeaDronesSee images (if accessible)
    images/aider/              ← AIDER emergency response images
    images/synthetic/          ← Programmatically generated labeled crops
    summit_detector/
      images/train/
      images/val/
      labels/train/            ← YOLO format: class_id cx cy w h (normalized)
      labels/val/
      summit_detector.yaml

Unified class taxonomy (must stay in sync with train_visual_detector.py):
  0  smoke            7  pipeline_damage   14 power_line_damage
  1  fire             8  chemical_plume    15 structural_crack
  2  fire_front       9  crop_disease      16 solar_defect
  3  person           10 pest_damage       17 dangerous_animal
  4  person_water     11 dry_field
  5  life_raft        12 vessel
  6  oil_spill        13 vessel_distress

Usage:
  python download_visual_datasets.py                          # download all sources + build
  python download_visual_datasets.py --source dfire coco_subset
  python download_visual_datasets.py --build-only            # rebuild combined dataset only
"""

import argparse
import json
import math
import os
import random
import shutil
import struct
import time
import urllib.error
import urllib.request
import zipfile
from typing import Dict, List, Optional, Tuple

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

OUTPUT_DIR = os.path.join(os.path.dirname(__file__), "data")
IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
SUMMIT_DIR = os.path.join(OUTPUT_DIR, "summit_detector")

# ---------------------------------------------------------------------------
# Unified class taxonomy
# ---------------------------------------------------------------------------

VISUAL_CLASSES: Dict[int, str] = {
    # Fire / Smoke
    0: "smoke",
    1: "fire",
    2: "fire_front",
    # SAR
    3: "person",
    4: "person_water",
    5: "life_raft",
    # Pipeline / Hazmat
    6: "oil_spill",
    7: "pipeline_damage",
    8: "chemical_plume",
    # Agricultural
    9: "crop_disease",
    10: "pest_damage",
    11: "dry_field",
    # Maritime
    12: "vessel",
    13: "vessel_distress",
    # Infrastructure
    14: "power_line_damage",
    15: "structural_crack",
    16: "solar_defect",
    # Wildlife
    17: "dangerous_animal",
}

# COCO category_id → VISUAL_CLASSES id.  Only categories we care about.
COCO_TO_VISUAL: Dict[int, int] = {
    1:  3,   # person           → person
    9:  12,  # boat             → vessel
    21: 17,  # cow              → dangerous_animal  (proxy for livestock / large animal)
    22: 17,  # elephant         → dangerous_animal
    23: 17,  # bear             → dangerous_animal
    24: 17,  # zebra            → dangerous_animal
    # The following are kept for scene-level usefulness but mapped to person / vessel
    3:  12,  # car              → vessel (surface vehicle proxy; filtered later)
    8:  12,  # truck            → vessel
    16: 17,  # bird             → dangerous_animal (raptor / UAV hazard proxy)
    17: 17,  # cat              → dangerous_animal
    18: 17,  # dog              → dangerous_animal
    19: 17,  # horse            → dangerous_animal
}

# ---------------------------------------------------------------------------
# HTTP helper (mirrors download_real_data.py pattern)
# ---------------------------------------------------------------------------

def _fetch(url: str, timeout: int = 60, retries: int = 3) -> Optional[bytes]:
    """Fetch a URL, returning raw bytes or None on failure."""
    headers = {"User-Agent": "Summit.OS/1.0 (public dataset research)"}
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read()
        except urllib.error.HTTPError as e:
            print(f"  HTTP {e.code}: {url}")
            return None
        except Exception as exc:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Attempt {attempt + 1} failed ({exc}), retrying in {wait}s…")
                time.sleep(wait)
            else:
                print(f"  Failed after {retries} attempts: {exc}")
                return None
    return None


def _fetch_stream(url: str, dest_path: str, timeout: int = 120) -> bool:
    """Stream-download a potentially large file to disk, showing progress."""
    headers = {"User-Agent": "Summit.OS/1.0 (public dataset research)"}
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            total = int(resp.getheader("Content-Length") or 0)
            downloaded = 0
            chunk = 1 << 16  # 64 KiB
            with open(dest_path, "wb") as f:
                while True:
                    block = resp.read(chunk)
                    if not block:
                        break
                    f.write(block)
                    downloaded += len(block)
                    if total:
                        pct = downloaded * 100 // total
                        print(f"\r  {pct:3d}%  {downloaded // 1048576} / {total // 1048576} MB", end="", flush=True)
            print()
        return True
    except Exception as exc:
        print(f"  Stream download failed: {exc}")
        return False


# ---------------------------------------------------------------------------
# Minimal PNG writer (stdlib only — no Pillow)
# Used by synthetic generator to write tiny solid-color patches.
# ---------------------------------------------------------------------------

def _write_png(path: str, width: int, height: int, rgb: Tuple[int, int, int]) -> None:
    """Write a single-color PNG using only stdlib (zlib + struct)."""
    import zlib

    def _u32be(n: int) -> bytes:
        return struct.pack(">I", n)

    def _chunk(tag: bytes, data: bytes) -> bytes:
        crc = zlib.crc32(tag + data) & 0xFFFFFFFF
        return _u32be(len(data)) + tag + data + _u32be(crc)

    # Build raw scanlines: filter byte (0) + RGB pixels
    row = bytes([0]) + bytes(rgb) * width
    raw = row * height
    compressed = zlib.compress(raw, 9)

    ihdr_data = (
        _u32be(width)
        + _u32be(height)
        + bytes([8, 2, 0, 0, 0])  # bit depth=8, colour=RGB, compression, filter, interlace
    )

    data = (
        b"\x89PNG\r\n\x1a\n"      # PNG signature
        + _chunk(b"IHDR", ihdr_data)
        + _chunk(b"IDAT", compressed)
        + _chunk(b"IEND", b"")
    )
    with open(path, "wb") as f:
        f.write(data)


# ---------------------------------------------------------------------------
# 1. D-Fire dataset  (fire / smoke, aerial imagery)
# ---------------------------------------------------------------------------

DFIRE_RELEASE_URL = (
    "https://github.com/gaiasd/DFireDataset/archive/refs/heads/master.zip"
)


def download_dfire() -> Dict[str, int]:
    """
    Download the D-Fire dataset from GitHub.

    D-Fire contains 21,527 images annotated for fire and smoke from aerial
    and ground-level cameras.  Annotations are in YOLO format with two classes:
      0 = fire, 1 = smoke

    We remap:  D-Fire 0 (fire) → VISUAL class 1 (fire)
               D-Fire 1 (smoke) → VISUAL class 0 (smoke)

    Saves to: packages/ml/data/images/fire_smoke/
    """
    dest = os.path.join(IMAGES_DIR, "fire_smoke")
    os.makedirs(dest, exist_ok=True)

    zip_path = os.path.join(OUTPUT_DIR, "_dfire_master.zip")

    print("Downloading D-Fire dataset (GitHub master archive)…")
    print(f"  URL: {DFIRE_RELEASE_URL}")

    if os.path.exists(zip_path):
        print("  Archive already cached, skipping download.")
    else:
        ok = _fetch_stream(DFIRE_RELEASE_URL, zip_path)
        if not ok:
            print(
                "\n  [MANUAL] D-Fire download failed. To install manually:\n"
                "    1. Visit https://github.com/gaiasd/DFireDataset\n"
                "    2. Click Code → Download ZIP\n"
                f"    3. Extract 'images/' and 'labels/' subdirectories into:\n"
                f"       {dest}\n"
            )
            return {}

    print("  Extracting archive…")
    counts: Dict[str, int] = {"smoke": 0, "fire": 0}
    images_extracted = 0
    labels_extracted = 0

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            members = zf.namelist()
            # D-Fire structure: DFireDataset-master/Train/images/, Train/labels/, Test/images/, Test/labels/
            for member in members:
                lower = member.lower()
                # Images
                if ("/images/" in lower or "/image/" in lower) and lower.endswith((".jpg", ".jpeg", ".png")):
                    fname = os.path.basename(member)
                    if not fname:
                        continue
                    # Preserve train/test split prefix in filename to avoid collisions
                    prefix = "train_" if "/train/" in lower else "test_"
                    out_path = os.path.join(dest, "images", prefix + fname)
                    os.makedirs(os.path.dirname(out_path), exist_ok=True)
                    with zf.open(member) as src, open(out_path, "wb") as tgt:
                        tgt.write(src.read())
                    images_extracted += 1
                # Labels (YOLO format)
                elif ("/labels/" in lower or "/label/" in lower) and lower.endswith(".txt"):
                    fname = os.path.basename(member)
                    if not fname:
                        continue
                    prefix = "train_" if "/train/" in lower else "test_"
                    label_path = os.path.join(dest, "labels", prefix + fname)
                    os.makedirs(os.path.dirname(label_path), exist_ok=True)
                    with zf.open(member) as src:
                        raw = src.read().decode("utf-8", errors="replace")
                    # Remap class IDs: D-Fire 0=fire→1, 1=smoke→0
                    remapped_lines = []
                    for line in raw.strip().splitlines():
                        parts = line.strip().split()
                        if not parts:
                            continue
                        try:
                            dfire_cls = int(parts[0])
                        except ValueError:
                            continue
                        visual_cls = 1 if dfire_cls == 0 else 0  # fire=1, smoke=0
                        counts["fire" if visual_cls == 1 else "smoke"] += 1
                        remapped_lines.append(
                            str(visual_cls) + " " + " ".join(parts[1:])
                        )
                    with open(label_path, "w") as lf:
                        lf.write("\n".join(remapped_lines) + "\n" if remapped_lines else "")
                    labels_extracted += 1

    except zipfile.BadZipFile as exc:
        print(f"  Bad zip file: {exc}. Removing cached archive.")
        os.remove(zip_path)
        return {}

    print(f"  Extracted {images_extracted} images, {labels_extracted} label files.")
    print(f"  Class counts — smoke: {counts['smoke']}, fire: {counts['fire']}")
    _print_summary("D-Fire", images_extracted, counts)
    return counts


# ---------------------------------------------------------------------------
# 2. COCO 2017 val subset
# ---------------------------------------------------------------------------

COCO_VAL_IMAGES_URL = "http://images.cocodataset.org/zips/val2017.zip"
COCO_VAL_ANNOT_URL  = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"


def download_coco_subset() -> Dict[str, int]:
    """
    Download COCO 2017 val images + annotations for relevant classes.

    Relevant COCO category IDs and their VISUAL_CLASSES mapping:
      person(1)→person(3), boat(9)→vessel(12), cow(21)/elephant(22)/
      bear(23)/zebra(24)→dangerous_animal(17), car(3)/truck(8)→vessel(12),
      bird(16)/cat(17)/dog(18)/horse(19)→dangerous_animal(17)

    COCO has 5,000 val images; we keep only images that contain at least one
    relevant category annotation.

    Saves to: packages/ml/data/images/coco_subset/
    """
    dest = os.path.join(IMAGES_DIR, "coco_subset")
    os.makedirs(dest, exist_ok=True)
    img_dir = os.path.join(dest, "images")
    lbl_dir = os.path.join(dest, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    annot_zip = os.path.join(OUTPUT_DIR, "_coco_annotations.zip")
    images_zip = os.path.join(OUTPUT_DIR, "_coco_val2017.zip")

    # --- Annotations ---
    if os.path.exists(annot_zip):
        print("  COCO annotations archive cached.")
    else:
        print(f"Downloading COCO 2017 annotations (~241 MB)…")
        if not _fetch_stream(COCO_VAL_ANNOT_URL, annot_zip, timeout=300):
            print(
                "\n  [MANUAL] COCO annotation download failed.\n"
                "    1. Visit https://cocodataset.org/#download\n"
                "    2. Download '2017 Val annotations [241MB]'\n"
                f"    3. Place the zip at: {annot_zip}\n"
            )
            return {}

    print("  Extracting COCO annotations…")
    annot_dir = os.path.join(OUTPUT_DIR, "_coco_annot")
    os.makedirs(annot_dir, exist_ok=True)
    with zipfile.ZipFile(annot_zip, "r") as zf:
        zf.extract("annotations/instances_val2017.json", annot_dir)

    annot_path = os.path.join(annot_dir, "annotations", "instances_val2017.json")
    print("  Parsing COCO annotations…")
    with open(annot_path) as f:
        coco = json.load(f)

    # Build {image_id: [{class_id, bbox:[x,y,w,h], img_w, img_h}]}
    id_to_info: Dict[int, Dict] = {
        img["id"]: {"file_name": img["file_name"], "w": img["width"], "h": img["height"]}
        for img in coco["images"]
    }

    relevant_image_ids = set()
    image_annotations: Dict[int, List[Dict]] = {}
    for ann in coco["annotations"]:
        cat_id = ann["category_id"]
        if cat_id not in COCO_TO_VISUAL:
            continue
        img_id = ann["image_id"]
        relevant_image_ids.add(img_id)
        if img_id not in image_annotations:
            image_annotations[img_id] = []
        x, y, bw, bh = ann["bbox"]
        image_annotations[img_id].append({
            "visual_cls": COCO_TO_VISUAL[cat_id],
            "x": x, "y": y, "bw": bw, "bh": bh,
        })

    print(f"  Found {len(relevant_image_ids)} relevant COCO val images.")

    # Write YOLO labels for these images (we can do this before downloading images)
    for img_id in relevant_image_ids:
        info = id_to_info[img_id]
        stem = os.path.splitext(info["file_name"])[0]
        lbl_path = os.path.join(lbl_dir, stem + ".txt")
        img_w, img_h = info["w"], info["h"]
        lines = []
        for ann in image_annotations[img_id]:
            # YOLO: cx cy w h normalized
            cx = (ann["x"] + ann["bw"] / 2) / img_w
            cy = (ann["y"] + ann["bh"] / 2) / img_h
            nw = ann["bw"] / img_w
            nh = ann["bh"] / img_h
            # Clamp to [0,1]
            cx = max(0.0, min(1.0, cx))
            cy = max(0.0, min(1.0, cy))
            nw = max(0.001, min(1.0, nw))
            nh = max(0.001, min(1.0, nh))
            lines.append(f"{ann['visual_cls']} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        with open(lbl_path, "w") as lf:
            lf.write("\n".join(lines) + "\n")

    # --- Images ---
    if os.path.exists(images_zip):
        print("  COCO val images archive cached.")
    else:
        print(f"Downloading COCO 2017 val images (~778 MB)…")
        print("  This is a large download. Press Ctrl-C to skip and place images manually.")
        print(f"  URL: {COCO_VAL_IMAGES_URL}")
        if not _fetch_stream(COCO_VAL_IMAGES_URL, images_zip, timeout=600):
            print(
                "\n  [MANUAL] COCO image download failed.\n"
                "    1. Visit https://cocodataset.org/#download\n"
                "    2. Download '2017 Val images [5K/1GB]'\n"
                f"    3. Extract val2017/ images into: {img_dir}\n"
                "  Labels have already been written; add images when available.\n"
            )
            return {}

    print("  Extracting relevant COCO images…")
    relevant_files = {id_to_info[iid]["file_name"] for iid in relevant_image_ids}
    extracted = 0
    with zipfile.ZipFile(images_zip, "r") as zf:
        for member in zf.namelist():
            fname = os.path.basename(member)
            if fname in relevant_files:
                out_path = os.path.join(img_dir, fname)
                if not os.path.exists(out_path):
                    with zf.open(member) as src, open(out_path, "wb") as tgt:
                        tgt.write(src.read())
                extracted += 1

    # Per-class counts
    counts: Dict[str, int] = {}
    for anns in image_annotations.values():
        for ann in anns:
            name = VISUAL_CLASSES[ann["visual_cls"]]
            counts[name] = counts.get(name, 0) + 1

    print(f"  Extracted {extracted} images.")
    _print_summary("COCO subset", extracted, counts)
    return counts


# ---------------------------------------------------------------------------
# 3. SeaDronesSee  (person-in-water from UAV)
# ---------------------------------------------------------------------------

SEADRONESSEE_BASE = "https://cloud.cs.uni-tuebingen.de/index.php/s/yMmqkNLMJKPHbx8/download"
SEADRONESSEE_GITHUB = "https://github.com/Ben93kie/SeaDronesSee"


def download_seadronessee() -> Dict[str, int]:
    """
    Attempt to download SeaDronesSee dataset for person-in-water detection from UAV.

    SeaDronesSee provides drone imagery with bounding box annotations for:
      swimmers (person_water), persons on boats, life rafts.

    The dataset is hosted at University of Tübingen. If direct download is
    unavailable, instructions are printed for manual acquisition.

    Saves to: packages/ml/data/images/seadronessee/
    """
    dest = os.path.join(IMAGES_DIR, "seadronessee")
    os.makedirs(dest, exist_ok=True)

    zip_path = os.path.join(OUTPUT_DIR, "_seadronessee.zip")

    print("Attempting SeaDronesSee download…")
    print(f"  Source: {SEADRONESSEE_GITHUB}")

    if os.path.exists(zip_path):
        print("  Archive cached.")
    else:
        ok = _fetch_stream(SEADRONESSEE_BASE, zip_path, timeout=300)
        if not ok:
            _print_manual_instructions(
                "SeaDronesSee (person-in-water, UAV)",
                steps=[
                    f"Visit: {SEADRONESSEE_GITHUB}",
                    "Follow dataset access instructions in the README.",
                    "Download the full dataset archive.",
                    f"Extract 'images/' and 'labels/' into: {dest}",
                    "Label format should be YOLO: class cx cy w h.",
                    "Remap SeaDronesSee class IDs to Summit.OS classes:",
                    "  swimmer/person_in_water → 4 (person_water)",
                    "  life_raft               → 5 (life_raft)",
                    "  person_on_board         → 3 (person)",
                ],
                dest=dest,
            )
            return {}

    print("  Extracting SeaDronesSee…")
    img_dir = os.path.join(dest, "images")
    lbl_dir = os.path.join(dest, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    images_count = 0
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.namelist():
                lower = member.lower()
                fname = os.path.basename(member)
                if not fname:
                    continue
                if lower.endswith((".jpg", ".jpeg", ".png")):
                    out = os.path.join(img_dir, fname)
                    with zf.open(member) as src, open(out, "wb") as tgt:
                        tgt.write(src.read())
                    images_count += 1
                elif lower.endswith(".txt"):
                    out = os.path.join(lbl_dir, fname)
                    with zf.open(member) as src, open(out, "wb") as tgt:
                        tgt.write(src.read())
    except zipfile.BadZipFile as exc:
        print(f"  Bad zip: {exc}")
        _print_manual_instructions(
            "SeaDronesSee",
            steps=[
                f"Visit: {SEADRONESSEE_GITHUB}",
                f"Download manually and extract to: {dest}",
            ],
            dest=dest,
        )
        return {}

    counts = {"person_water": images_count}  # rough approximation
    _print_summary("SeaDronesSee", images_count, counts)
    return counts


# ---------------------------------------------------------------------------
# 4. AIDER  (Aerial Image Dataset for Emergency Response)
# ---------------------------------------------------------------------------

AIDER_GITHUB = "https://github.com/ckyrkou/AIDER"
AIDER_ARCHIVE_URL = "https://github.com/ckyrkou/AIDER/archive/refs/heads/master.zip"


def download_aerial_sar() -> Dict[str, int]:
    """
    Download AIDER — Aerial Image Dataset for Emergency Response.

    AIDER contains aerial imagery of four disaster types:
      fire/smoke, flood, collapsed building/rubble, normal (background).

    AIDER images do NOT include bounding box annotations, only scene-level
    class labels. We convert these to whole-image YOLO pseudo-annotations
    (bbox covering 80% of the frame) suitable for training a detector backbone.

    Class mapping:
      AIDER fire/smoke → VISUAL 1 (fire) + 0 (smoke)
      AIDER flood      → (no direct class; used as background diversity)
      AIDER collapsed  → VISUAL 15 (structural_crack)

    Saves to: packages/ml/data/images/aider/
    """
    dest = os.path.join(IMAGES_DIR, "aider")
    os.makedirs(dest, exist_ok=True)
    img_dir = os.path.join(dest, "images")
    lbl_dir = os.path.join(dest, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    zip_path = os.path.join(OUTPUT_DIR, "_aider_master.zip")

    print("Downloading AIDER dataset (Aerial Image Dataset for Emergency Response)…")
    print(f"  Source: {AIDER_GITHUB}")

    if os.path.exists(zip_path):
        print("  Archive cached.")
    else:
        ok = _fetch_stream(AIDER_ARCHIVE_URL, zip_path, timeout=300)
        if not ok:
            _print_manual_instructions(
                "AIDER (Aerial Emergency Response)",
                steps=[
                    f"Visit: {AIDER_GITHUB}",
                    "Click Code → Download ZIP.",
                    f"Extract into: {dest}",
                    "Expected subdirectories: fire_smoke/, flood/, collapsed/, normal/",
                ],
                dest=dest,
            )
            return {}

    print("  Extracting AIDER…")

    # AIDER GitHub repo structure: AIDER-master/{fire_smoke,flood,collapsed,normal}/...
    aider_class_map = {
        "fire_smoke": (1, "fire"),
        "collapsed":  (15, "structural_crack"),
        "flood":      (None, None),   # background diversity — no annotation
    }

    counts: Dict[str, int] = {}
    images_extracted = 0

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for member in zf.namelist():
                lower = member.lower()
                if not lower.endswith((".jpg", ".jpeg", ".png")):
                    continue
                fname = os.path.basename(member)
                if not fname:
                    continue

                # Determine AIDER class from directory path
                aider_cls = None
                for cls_name in aider_class_map:
                    if f"/{cls_name}/" in lower or lower.startswith(cls_name + "/"):
                        aider_cls = cls_name
                        break

                if aider_cls is None:
                    continue

                visual_cls_id, visual_cls_name = aider_class_map[aider_cls]
                safe_name = aider_cls + "_" + fname
                out_img = os.path.join(img_dir, safe_name)
                with zf.open(member) as src, open(out_img, "wb") as tgt:
                    tgt.write(src.read())

                if visual_cls_id is not None:
                    # Whole-image pseudo-annotation (80% coverage, centered)
                    stem = os.path.splitext(safe_name)[0]
                    lbl_path = os.path.join(lbl_dir, stem + ".txt")
                    with open(lbl_path, "w") as lf:
                        lf.write(f"{visual_cls_id} 0.500000 0.500000 0.800000 0.800000\n")
                    counts[visual_cls_name] = counts.get(visual_cls_name, 0) + 1

                images_extracted += 1

    except zipfile.BadZipFile as exc:
        print(f"  Bad zip: {exc}")
        return {}

    _print_summary("AIDER", images_extracted, counts)
    return counts


# ---------------------------------------------------------------------------
# 5. Synthetic labeled crops
#    For: pipeline, agricultural, chemical, infrastructure, maritime distress
# ---------------------------------------------------------------------------

# Domain definitions: (visual_class_id, primary_rgb, secondary_rgb, description)
_SYNTHETIC_DOMAINS = [
    # Pipeline / Hazmat
    (6,  (40,  60,  20),  (180, 120,  30), "oil_spill",        "Dark oily sheen patches on soil/water"),
    (7,  (80,  80,  80),  (200,  50,  50), "pipeline_damage",  "Rust-coloured breach on grey pipe cross-section"),
    (8,  (200, 230, 200), (120, 200, 100), "chemical_plume",   "Pale yellow-green smoke/plume pattern"),
    # Agricultural
    (9,  (100, 140,  40), (200,  80,  80), "crop_disease",     "Brown/red lesion patches on green canopy"),
    (10, (120, 160,  50), (180, 120,  20), "pest_damage",      "Irregular brown spots on crop canopy"),
    (11, (180, 160,  80), (220, 200, 100), "dry_field",        "Uniform straw-yellow parched soil"),
    # Maritime distress
    (13, (200,  60,  60), (255, 120,  30), "vessel_distress",  "Red/orange distress signal flare on dark water"),
    # Infrastructure
    (14, (60,   60,  60), (200,  40,  40), "power_line_damage","Burned/broken conductor segment"),
    (15, (150, 150, 150), (80,   80,  80), "structural_crack", "Linear crack on concrete surface"),
    (16, (240, 200,  20), (80,   80,  60), "solar_defect",     "Dark cell patch on bright panel surface"),
]

# Sizes of synthetic images and bounding boxes
_IMG_W = 416
_IMG_H = 416
_MIN_BOX_FRAC = 0.10   # minimum box side as fraction of image
_MAX_BOX_FRAC = 0.50   # maximum box side


def _make_synthetic_image(
    img_path: str,
    lbl_path: str,
    visual_cls: int,
    bg_rgb: Tuple[int, int, int],
    fg_rgb: Tuple[int, int, int],
    n_boxes: int = 1,
    rng: Optional[random.Random] = None,
) -> None:
    """
    Write a synthetic PNG with solid-color bounding-box regions.

    Background = bg_rgb, bounding box fills = fg_rgb with slight noise per
    box (simulated by varying shade).  Saves YOLO label alongside.
    """
    import zlib

    if rng is None:
        rng = random.Random()

    W, H = _IMG_W, _IMG_H

    # Build pixel buffer: row-major, RGB
    pixels = list(bg_rgb) * (W * H)  # flat list of R,G,B,...

    yolo_lines = []
    for _ in range(n_boxes):
        min_bw = int(_MIN_BOX_FRAC * W)
        max_bw = int(_MAX_BOX_FRAC * W)
        min_bh = int(_MIN_BOX_FRAC * H)
        max_bh = int(_MAX_BOX_FRAC * H)

        bw = rng.randint(min_bw, max_bw)
        bh = rng.randint(min_bh, max_bh)
        x0 = rng.randint(0, W - bw)
        y0 = rng.randint(0, H - bh)

        # Fill box with fg_rgb (slight brightness variation per pixel for realism)
        for row in range(y0, y0 + bh):
            for col in range(x0, x0 + bw):
                noise = rng.randint(-15, 15)
                r = max(0, min(255, fg_rgb[0] + noise))
                g = max(0, min(255, fg_rgb[1] + noise))
                b = max(0, min(255, fg_rgb[2] + noise))
                idx = (row * W + col) * 3
                pixels[idx]     = r
                pixels[idx + 1] = g
                pixels[idx + 2] = b

        cx = (x0 + bw / 2) / W
        cy = (y0 + bh / 2) / H
        nw = bw / W
        nh = bh / H
        yolo_lines.append(f"{visual_cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

    # Encode as PNG (filter byte 0 = None per scanline)
    def _u32be(n: int) -> bytes:
        return struct.pack(">I", n)

    def _chunk(tag: bytes, data: bytes) -> bytes:
        crc = zlib.crc32(tag + data) & 0xFFFFFFFF
        return _u32be(len(data)) + tag + data + _u32be(crc)

    raw_rows = bytearray()
    for row in range(H):
        raw_rows.append(0)  # filter type None
        base = row * W * 3
        raw_rows.extend(pixels[base: base + W * 3])

    compressed = zlib.compress(bytes(raw_rows), 1)  # level 1 = fast

    ihdr = _u32be(W) + _u32be(H) + bytes([8, 2, 0, 0, 0])
    png = (
        b"\x89PNG\r\n\x1a\n"
        + _chunk(b"IHDR", ihdr)
        + _chunk(b"IDAT", compressed)
        + _chunk(b"IEND", b"")
    )

    with open(img_path, "wb") as f:
        f.write(png)

    with open(lbl_path, "w") as f:
        f.write("\n".join(yolo_lines) + "\n")


def generate_synthetic_crops(n_per_class: int = 200, seed: int = 42) -> Dict[str, int]:
    """
    Generate synthetic labeled images for domains with no suitable public dataset.

    Each image is a solid-color background with one or more solid-color
    foreground bounding boxes.  This is intentionally simple — its purpose
    is to give the detector a weak signal for rare classes while real data
    collection proceeds.  For production, replace with domain-expert labeled
    aerial imagery.

    n_per_class: number of synthetic images per class (default 200)
    seed: random seed for reproducibility

    Saves to: packages/ml/data/images/synthetic/
    """
    dest = os.path.join(IMAGES_DIR, "synthetic")
    img_dir = os.path.join(dest, "images")
    lbl_dir = os.path.join(dest, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    print(f"Generating {n_per_class} synthetic images per class ({len(_SYNTHETIC_DOMAINS)} classes)…")

    rng = random.Random(seed)
    counts: Dict[str, int] = {}
    total_images = 0

    for visual_cls, fg_rgb, bg_rgb, cls_name, description in _SYNTHETIC_DOMAINS:
        print(f"  {cls_name}: {description}")
        for i in range(n_per_class):
            fname = f"{cls_name}_{i:05d}.png"
            img_path = os.path.join(img_dir, fname)
            lbl_path = os.path.join(lbl_dir, f"{cls_name}_{i:05d}.txt")

            # Vary number of boxes: mostly 1, sometimes 2-3 for realism
            n_boxes = rng.choices([1, 2, 3], weights=[0.70, 0.20, 0.10])[0]

            # Slight background color variation
            bg_var = tuple(max(0, min(255, c + rng.randint(-20, 20))) for c in bg_rgb)

            _make_synthetic_image(
                img_path, lbl_path,
                visual_cls=visual_cls,
                bg_rgb=bg_var,     # type: ignore[arg-type]
                fg_rgb=fg_rgb,
                n_boxes=n_boxes,
                rng=rng,
            )
            total_images += 1

        counts[cls_name] = n_per_class

    _print_summary("Synthetic", total_images, counts)
    return counts


# ---------------------------------------------------------------------------
# 6. Build combined YOLO dataset
# ---------------------------------------------------------------------------

def build_combined_dataset(val_fraction: float = 0.20, seed: int = 42) -> None:
    """
    Merge all downloaded sources into a unified YOLO-format directory:

      packages/ml/data/summit_detector/
        images/train/
        images/val/
        labels/train/
        labels/val/
        summit_detector.yaml

    Only images that have a corresponding label file (non-empty) are included.
    Performs an 80/20 train/val stratified split.
    """
    print("\n" + "=" * 60)
    print("Building combined Summit.OS detector dataset…")
    print("=" * 60)

    # Collect all (image_path, label_path) pairs from all sources
    sources = [
        os.path.join(IMAGES_DIR, "fire_smoke"),
        os.path.join(IMAGES_DIR, "coco_subset"),
        os.path.join(IMAGES_DIR, "seadronessee"),
        os.path.join(IMAGES_DIR, "aider"),
        os.path.join(IMAGES_DIR, "synthetic"),
    ]

    all_pairs: List[Tuple[str, str]] = []
    for src in sources:
        img_dir = os.path.join(src, "images")
        lbl_dir = os.path.join(src, "labels")
        if not os.path.isdir(img_dir) or not os.path.isdir(lbl_dir):
            continue
        for fname in os.listdir(img_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            stem = os.path.splitext(fname)[0]
            lbl_path = os.path.join(lbl_dir, stem + ".txt")
            if not os.path.exists(lbl_path):
                continue
            # Skip empty label files (background-only images we don't want)
            if os.path.getsize(lbl_path) == 0:
                continue
            all_pairs.append((os.path.join(img_dir, fname), lbl_path))

    print(f"  Found {len(all_pairs)} labeled images across all sources.")

    if not all_pairs:
        print("  No labeled images found. Run downloads first.")
        return

    # Shuffle then split
    rng = random.Random(seed)
    rng.shuffle(all_pairs)
    n_val = max(1, int(len(all_pairs) * val_fraction))
    val_pairs  = all_pairs[:n_val]
    train_pairs = all_pairs[n_val:]

    print(f"  Split: {len(train_pairs)} train / {len(val_pairs)} val")

    # Create output dirs
    for split in ("train", "val"):
        os.makedirs(os.path.join(SUMMIT_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(SUMMIT_DIR, "labels", split), exist_ok=True)

    def _copy_pair(img_src: str, lbl_src: str, split: str) -> None:
        img_fname = os.path.basename(img_src)
        lbl_fname = os.path.splitext(img_fname)[0] + ".txt"
        shutil.copy2(img_src, os.path.join(SUMMIT_DIR, "images", split, img_fname))
        shutil.copy2(lbl_src, os.path.join(SUMMIT_DIR, "labels", split, lbl_fname))

    for img_src, lbl_src in train_pairs:
        _copy_pair(img_src, lbl_src, "train")
    for img_src, lbl_src in val_pairs:
        _copy_pair(img_src, lbl_src, "val")

    # Write summit_detector.yaml
    yaml_path = os.path.join(SUMMIT_DIR, "summit_detector.yaml")
    class_names = [VISUAL_CLASSES[i] for i in sorted(VISUAL_CLASSES)]
    yaml_lines = [
        "# Summit.OS unified visual detector dataset",
        f"# Auto-generated by download_visual_datasets.py",
        "",
        f"path: {SUMMIT_DIR}",
        "train: images/train",
        "val: images/val",
        "",
        f"nc: {len(VISUAL_CLASSES)}",
        "names:",
    ]
    for i, name in enumerate(class_names):
        yaml_lines.append(f"  {i}: {name}")

    with open(yaml_path, "w") as f:
        f.write("\n".join(yaml_lines) + "\n")

    print(f"  Wrote: {yaml_path}")

    # Class distribution summary
    class_counts: Dict[int, int] = {i: 0 for i in VISUAL_CLASSES}
    for _, lbl_path in (train_pairs + val_pairs):
        with open(lbl_path) as lf:
            for line in lf:
                parts = line.strip().split()
                if parts:
                    try:
                        cls_id = int(parts[0])
                        if cls_id in class_counts:
                            class_counts[cls_id] += 1
                    except ValueError:
                        pass

    print("\n  Class distribution in combined dataset:")
    print(f"  {'ID':<4} {'Name':<22} {'Annotations':>12}")
    print("  " + "-" * 40)
    for cls_id in sorted(VISUAL_CLASSES):
        name = VISUAL_CLASSES[cls_id]
        cnt = class_counts.get(cls_id, 0)
        marker = " *** EMPTY" if cnt == 0 else ""
        print(f"  {cls_id:<4} {name:<22} {cnt:>12,}{marker}")
    print()

    total_ann = sum(class_counts.values())
    print(f"  Total annotations: {total_ann:,}")
    print(f"  Dataset root: {SUMMIT_DIR}")
    print(f"  YAML config:  {yaml_path}")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _print_summary(source: str, n_images: int, counts: Dict[str, int]) -> None:
    print(f"\n  [{source}] Summary:")
    print(f"    Images: {n_images:,}")
    if counts:
        print(f"    Labels per class:")
        for name, cnt in sorted(counts.items(), key=lambda kv: -kv[1]):
            print(f"      {name:<25} {cnt:>8,}")
    print()


def _print_manual_instructions(dataset: str, steps: List[str], dest: str) -> None:
    print(f"\n  [MANUAL REQUIRED] {dataset} cannot be auto-downloaded.")
    print(f"  Follow these steps to add it manually:")
    for i, step in enumerate(steps, 1):
        print(f"    {i}. {step}")
    print(f"  Target directory: {dest}")
    print()


# ---------------------------------------------------------------------------
# Source registry
# ---------------------------------------------------------------------------

SOURCES = {
    "dfire":            download_dfire,
    "coco_subset":      download_coco_subset,
    "seadronessee":     download_seadronessee,
    "aerial_sar":       download_aerial_sar,
    "synthetic":        generate_synthetic_crops,
}

# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Download and build visual object detection datasets for Summit.OS.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument(
        "--source",
        nargs="+",
        choices=list(SOURCES.keys()),
        metavar="SOURCE",
        help=(
            "Download only specific source(s). "
            f"Choices: {', '.join(SOURCES)}. "
            "Default: all."
        ),
    )
    parser.add_argument(
        "--build-only",
        action="store_true",
        help="Skip all downloads; just rebuild the combined dataset from existing downloads.",
    )
    parser.add_argument(
        "--val-fraction",
        type=float,
        default=0.20,
        help="Fraction of data to use for validation (default: 0.20).",
    )
    parser.add_argument(
        "--synthetic-n",
        type=int,
        default=200,
        help="Number of synthetic images per class (default: 200).",
    )
    args = parser.parse_args()

    # Ensure base directories exist
    for d in (OUTPUT_DIR, IMAGES_DIR, SUMMIT_DIR):
        os.makedirs(d, exist_ok=True)

    if args.build_only:
        build_combined_dataset(val_fraction=args.val_fraction)
        return

    sources_to_run = args.source if args.source else list(SOURCES.keys())

    print(f"Summit.OS Visual Dataset Downloader")
    print(f"Output root : {OUTPUT_DIR}")
    print(f"Sources     : {', '.join(sources_to_run)}")
    print()

    for source_name in sources_to_run:
        fn = SOURCES[source_name]
        print(f"{'─' * 60}")
        print(f"Source: {source_name}")
        print(f"{'─' * 60}")
        if source_name == "synthetic":
            fn(n_per_class=args.synthetic_n)    # type: ignore[call-arg]
        else:
            fn()

    build_combined_dataset(val_fraction=args.val_fraction)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/train_visual_detector.py
"""
Fine-tune YOLOv8n on the Summit.OS visual dataset for multi-domain object detection.

Covers all operational domains out of the box:
  Wildfire       — smoke, fire, fire_front
  Search & Rescue — person, person_water, life_raft
  Oil / Hazmat   — oil_spill, pipeline_damage, chemical_plume
  Agriculture    — crop_disease, pest_damage, dry_field
  Maritime       — vessel, vessel_distress
  Infrastructure — power_line_damage, structural_crack, solar_defect
  Wildlife       — dangerous_animal

Prerequisites:
  pip install ultralytics onnx onnxruntime

Usage:
  # Build dataset first (if not already done)
  python download_visual_datasets.py

  # Train (default: YOLOv8n, 100 epochs, all domains)
  python train_visual_detector.py

  # Quick smoke-test (5 epochs, CPU-safe)
  python train_visual_detector.py --epochs 5 --device cpu

  # Full training with GPU
  python train_visual_detector.py --epochs 200 --device 0

  # Export only (skip training, export existing checkpoint)
  python train_visual_detector.py --export-only --checkpoint runs/detect/summit_detector/weights/best.pt

Output:
  packages/ml/models/summit_detector.onnx   (dropped in place — inference service hot-swaps)
  packages/ml/models/summit_detector_classes.json
"""

import argparse
import json
import os
import sys
from pathlib import Path

# ── paths ─────────────────────────────────────────────────────────────────────
SCRIPT_DIR  = Path(__file__).parent
DATA_DIR    = SCRIPT_DIR / "data"
MODELS_DIR  = SCRIPT_DIR / "models"
DATASET_DIR = DATA_DIR / "summit_detector"
YAML_PATH   = DATASET_DIR / "summit_detector.yaml"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── class taxonomy ─────────────────────────────────────────────────────────────
# Must stay in sync with download_visual_datasets.py VISUAL_CLASSES
VISUAL_CLASSES = {
    0:  "smoke",
    1:  "fire",
    2:  "fire_front",
    3:  "person",
    4:  "person_water",
    5:  "life_raft",
    6:  "oil_spill",
    7:  "pipeline_damage",
    8:  "chemical_plume",
    9:  "crop_disease",
    10: "pest_damage",
    11: "dry_field",
    12: "vessel",
    13: "vessel_distress",
    14: "power_line_damage",
    15: "structural_crack",
    16: "solar_defect",
    17: "dangerous_animal",
}

# Domain groupings — used for per-domain augmentation strategies
DOMAIN_CLASSES = {
    "wildfire":        [0, 1, 2],           # smoke, fire, fire_front
    "search_rescue":   [3, 4, 5],           # person, person_water, life_raft
    "hazmat_pipeline": [6, 7, 8],           # oil_spill, pipeline_damage, chemical_plume
    "agriculture":     [9, 10, 11],         # crop_disease, pest_damage, dry_field
    "maritime":        [12, 13],            # vessel, vessel_distress
    "infrastructure":  [14, 15, 16],        # power_line_damage, structural_crack, solar_defect
    "wildlife":        [17],                # dangerous_animal
}


# ── augmentation profiles ──────────────────────────────────────────────────────
def _augmentation_kwargs() -> dict:
    """
    Augmentation settings tuned for aerial/UAV imagery across all operational domains.

    Key choices:
      - High HSV-Hue shift: fire and hazmat plumes have inconsistent colour rendering
        across different sensor types and times of day.
      - Mosaic=1.0: forces the model to learn partial objects (critical for smoke at edge
        of frame, partial persons in water, partially visible vessels).
      - Flipud=0.5: UAV downward-looking imagery has no canonical 'up'.
      - Degrees=30: rotational invariance for top-down infrastructure inspection.
      - Scale=0.6: handles extreme altitude variation (drone height 30m–300m).
      - Perspective=0.001: simulates lens tilt on oblique aerial angles.
    """
    return dict(
        hsv_h=0.02,        # hue jitter — fire/hazmat colour variance
        hsv_s=0.7,         # saturation — smoke opacity varies
        hsv_v=0.4,         # value — altitude/lighting variation
        degrees=30,        # rotation — top-down has no canonical orientation
        translate=0.1,
        scale=0.6,         # extreme zoom range for altitude variance
        shear=0.0,
        perspective=0.001, # oblique UAV angles
        flipud=0.5,        # UAV: vertical flip is valid
        fliplr=0.5,
        mosaic=1.0,        # always-on: critical for partial objects
        mixup=0.1,         # gentle mixup for domain blending
        copy_paste=0.1,    # copy-paste augmentation for rare classes
    )


# ── dataset yaml guard ─────────────────────────────────────────────────────────
def _ensure_dataset() -> None:
    if not YAML_PATH.exists():
        print(
            f"\n[ERROR] Dataset YAML not found at {YAML_PATH}\n"
            "Run first:\n"
            "  python download_visual_datasets.py\n"
        )
        sys.exit(1)

    # Check train/val image dirs are non-empty
    for split in ("train", "val"):
        img_dir = DATASET_DIR / "images" / split
        if not img_dir.exists() or not any(img_dir.iterdir()):
            print(
                f"\n[ERROR] {img_dir} is empty.\n"
                "Run first:\n"
                "  python download_visual_datasets.py\n"
            )
            sys.exit(1)

    # Count images
    train_count = len(list((DATASET_DIR / "images" / "train").iterdir()))
    val_count   = len(list((DATASET_DIR / "images" / "val").iterdir()))
    print(f"[dataset] train={train_count} images  val={val_count} images")


# ── training ───────────────────────────────────────────────────────────────────
def train(
    epochs: int,
    device: str,
    batch: int,
    imgsz: int,
    model_size: str,
    project: str,
    name: str,
    resume: bool,
    patience: int,
) -> Path:
    """
    Fine-tune YOLOv8 on summit_detector dataset.
    Returns path to the best checkpoint (.pt).
    """
    try:
        from ultralytics import YOLO
    except ImportError:
        print("[ERROR] ultralytics not installed.\n  pip install ultralytics")
        sys.exit(1)

    base_model = f"yolov8{model_size}.pt"
    print(f"\n[train] base={base_model}  epochs={epochs}  device={device}  imgsz={imgsz}")
    print(f"[train] dataset={YAML_PATH}")

    model = YOLO(base_model)

    aug = _augmentation_kwargs()

    results = model.train(
        data=str(YAML_PATH),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        device=device,
        project=project,
        name=name,
        resume=resume,
        patience=patience,
        save=True,
        save_period=10,         # checkpoint every 10 epochs
        val=True,
        plots=True,
        # Augmentation
        **aug,
        # Loss weights — up-weight localization for small aerial targets
        box=7.5,
        cls=0.5,
        dfl=1.5,
        # Optimizer
        optimizer="AdamW",
        lr0=1e-3,
        lrf=0.01,               # final LR = lr0 * lrf
        warmup_epochs=3,
        weight_decay=5e-4,
        # Misc
        workers=min(8, os.cpu_count() or 4),
        seed=42,
        verbose=True,
    )

    best_pt = Path(project) / name / "weights" / "best.pt"
    if not best_pt.exists():
        # Fallback: find in ultralytics default run dir
        import glob
        candidates = glob.glob(f"{project}/{name}*/weights/best.pt")
        if candidates:
            best_pt = Path(sorted(candidates)[-1])

    print(f"\n[train] best checkpoint: {best_pt}")
    return best_pt


# ── export to ONNX ─────────────────────────────────────────────────────────────
def export_onnx(checkpoint: Path, imgsz: int) -> Path:
    """
    Export YOLOv8 checkpoint to ONNX and copy to models/ for hot-swap.
    """
    try:
        from ultralytics import YOLO
    except ImportError:
        print("[ERROR] ultralytics not installed.\n  pip install ultralytics")
        sys.exit(1)

    print(f"\n[export] {checkpoint} → ONNX (imgsz={imgsz})")
    model = YOLO(str(checkpoint))

    # Export — ultralytics writes .onnx alongside the .pt file
    model.export(
        format="onnx",
        imgsz=imgsz,
        dynamic=False,          # static shapes for ONNX Runtime compatibility
        simplify=True,          # onnx-simplifier reduces node count
        opset=17,               # opset 17 = broad ONNX Runtime support
        half=False,             # FP32 — maximises CPU compatibility for edge devices
    )

    exported_onnx = checkpoint.with_suffix(".onnx")
    dest_onnx     = MODELS_DIR / "summit_detector.onnx"

    if not exported_onnx.exists():
        print(f"[WARN] ONNX export not found at {exported_onnx} — check ultralytics output")
    else:
        import shutil
        shutil.copy2(exported_onnx, dest_onnx)
        size_mb = dest_onnx.stat().st_size / 1e6
        print(f"[export] {dest_onnx}  ({size_mb:.1f} MB)")

    # Write class map for the inference service
    classes_path = MODELS_DIR / "summit_detector_classes.json"
    with open(classes_path, "w") as f:
        json.dump(VISUAL_CLASSES, f, indent=2)
    print(f"[export] {classes_path}")

    return dest_onnx


# ── quick validation ───────────────────────────────────────────────────────────
def validate(checkpoint: Path, imgsz: int) -> None:
    """Run val loop and print per-class mAP50."""
    try:
        from ultralytics import YOLO
    except ImportError:
        return

    print(f"\n[validate] {checkpoint}")
    model   = YOLO(str(checkpoint))
    metrics = model.val(data=str(YAML_PATH), imgsz=imgsz, verbose=True)

    print("\n[validate] Per-class mAP50:")
    if hasattr(metrics, "ap_class_index") and hasattr(metrics, "box"):
        for i, cls_idx in enumerate(metrics.ap_class_index):
            cls_name = VISUAL_CLASSES.get(int(cls_idx), str(cls_idx))
            ap50     = float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else float("nan")
            print(f"  {cls_name:25s}  mAP50={ap50:.3f}")

    map50 = getattr(getattr(metrics, "box", None), "map50", None)
    if map50 is not None:
        print(f"\n[validate] Overall mAP50={float(map50):.3f}")


# ── ONNX smoke-test ────────────────────────────────────────────────────────────
def smoke_test_onnx(onnx_path: Path, imgsz: int) -> None:
    """
    Verify the exported ONNX model loads and runs a single forward pass.
    No real image needed — random tensor is sufficient for shape/type validation.
    """
    try:
        import numpy as np
        import onnxruntime as ort
    except ImportError:
        print("[smoke-test] onnxruntime/numpy not available — skipping")
        return

    if not onnx_path.exists():
        print(f"[smoke-test] {onnx_path} not found — skipping")
        return

    print(f"\n[smoke-test] {onnx_path}")
    sess    = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    inp     = sess.get_inputs()[0]
    dummy   = np.random.rand(1, 3, imgsz, imgsz).astype(np.float32)
    outputs = sess.run(None, {inp.name: dummy})
    print(f"[smoke-test] output shapes: {[o.shape for o in outputs]}")
    print("[smoke-test] PASS")


# ── CLI ────────────────────────────────────────────────────────────────────────
def main() -> None:
    parser = argparse.ArgumentParser(
        description="Train Summit.OS visual detector (YOLOv8) across all operational domains"
    )
    parser.add_argument("--epochs",       type=int,   default=100,
                        help="Training epochs (default 100; use 5 for quick smoke-test)")
    parser.add_argument("--device",       type=str,   default="",
                        help="'cpu', '0', '0,1', 'mps' — empty = auto-detect")
    parser.add_argument("--batch",        type=int,   default=16,
                        help="Batch size (reduce if OOM; -1 = auto)")
    parser.add_argument("--imgsz",        type=int,   default=640,
                        help="Input image size (default 640)")
    parser.add_argument("--model",        type=str,   default="n",
                        choices=["n", "s", "m", "l", "x"],
                        help="YOLOv8 model size: n=nano, s=small, m=medium (default: n)")
    parser.add_argument("--project",      type=str,   default="runs/detect",
                        help="Output directory for training runs")
    parser.add_argument("--name",         type=str,   default="summit_detector",
                        help="Run name within --project")
    parser.add_argument("--resume",       action="store_true",
                        help="Resume interrupted training from last checkpoint")
    parser.add_argument("--patience",     type=int,   default=50,
                        help="Early-stopping patience in epochs (0 = disabled)")
    parser.add_argument("--export-only",  action="store_true",
                        help="Skip training — export existing checkpoint to ONNX")
    parser.add_argument("--checkpoint",   type=str,   default=None,
                        help="Path to .pt checkpoint for --export-only or post-train export")
    parser.add_argument("--no-validate",  action="store_true",
                        help="Skip validation after training")
    parser.add_argument("--no-smoke-test", action="store_true",
                        help="Skip ONNX smoke-test after export")
    parser.add_argument("--list-classes", action="store_true",
                        help="Print class taxonomy and exit")
    args = parser.parse_args()

    if args.list_classes:
        print("Summit.OS visual detector — 18 classes:\n")
        for idx, name in VISUAL_CLASSES.items():
            domain = next(
                (d for d, ids in DOMAIN_CLASSES.items() if idx in ids), "—"
            )
            print(f"  {idx:2d}  {name:25s}  [{domain}]")
        return

    # ── export-only mode ────────────────────────────────────────────────────
    if args.export_only:
        if not args.checkpoint:
            print("[ERROR] --export-only requires --checkpoint <path/to/best.pt>")
            sys.exit(1)
        ckpt = Path(args.checkpoint)
        if not ckpt.exists():
            print(f"[ERROR] checkpoint not found: {ckpt}")
            sys.exit(1)
        onnx_path = export_onnx(ckpt, args.imgsz)
        if not args.no_smoke_test:
            smoke_test_onnx(onnx_path, args.imgsz)
        return

    # ── full training ───────────────────────────────────────────────────────
    _ensure_dataset()

    best_pt = train(
        epochs   = args.epochs,
        device   = args.device,
        batch    = args.batch,
        imgsz    = args.imgsz,
        model_size = args.model,
        project  = args.project,
        name     = args.name,
        resume   = args.resume,
        patience = args.patience,
    )

    # Use user-supplied checkpoint if provided (e.g. after manual export decision)
    if args.checkpoint:
        best_pt = Path(args.checkpoint)

    if not args.no_validate and best_pt.exists():
        validate(best_pt, args.imgsz)

    onnx_path = export_onnx(best_pt, args.imgsz)

    if not args.no_smoke_test:
        smoke_test_onnx(onnx_path, args.imgsz)

    print(f"""
╔══════════════════════════════════════════════════════════╗
║  Summit.OS visual detector training complete             ║
║                                                          ║
║  Model   : packages/ml/models/summit_detector.onnx       ║
║  Classes : packages/ml/models/summit_detector_classes.json║
║                                                          ║
║  Inference service will hot-swap automatically.          ║
║  Restart apps/inference to pick up new model.            ║
╚══════════════════════════════════════════════════════════╝
""")


if __name__ == "__main__":
    main()


## Step 4 — Download and Prepare Datasets

In [ ]:
import os
os.chdir('/content')
print("Working directory:", os.getcwd())

In [ ]:
# Download all public datasets (D-Fire, COCO subset) and generate synthetic data.
# D-Fire: ~600 MB download from GitHub
# COCO subset: ~1 GB download
# Synthetic: generated locally, no download needed
# Total estimated time: 15-30 min depending on network speed
!python download_visual_datasets.py

## Step 5 — Train the Visual Detector

In [ ]:
# Fine-tune YOLOv8n for 100 epochs on the combined dataset.
# Estimated time on T4 GPU: 45–90 min depending on dataset size.
# The script will periodically checkpoint (every 10 epochs) and validate.
!python train_visual_detector.py --epochs 100 --device cuda

## Step 6 — Verify Outputs

In [ ]:
import os

outputs = [
    '/content/models/summit_detector.onnx',
    '/content/models/summit_detector_classes.json',
]

print("Output file sizes:")
for path in outputs:
    if os.path.exists(path):
        size = os.path.getsize(path)
        if size >= 1_000_000:
            print(f"  {path}  ->  {size / 1e6:.1f} MB")
        else:
            print(f"  {path}  ->  {size / 1024:.1f} KB")
    else:
        print(f"  MISSING: {path}")

# Also print training run directory sizes if present
run_dir = '/content/runs/detect/summit_detector'
if os.path.isdir(run_dir):
    weights_dir = os.path.join(run_dir, 'weights')
    if os.path.isdir(weights_dir):
        print("\nCheckpoints in runs/detect/summit_detector/weights/:")
        for fname in sorted(os.listdir(weights_dir)):
            fpath = os.path.join(weights_dir, fname)
            size = os.path.getsize(fpath)
            print(f"  {fname}  ->  {size / 1e6:.1f} MB")

## Step 7 — Download Model Files

Running the cell below will trigger two browser download prompts.
Save both files to your local machine, then place them in `packages/ml/models/`.
The inference service will hot-swap the new model on next restart.

In [ ]:
from google.colab import files

files.download('/content/models/summit_detector.onnx')
files.download('/content/models/summit_detector_classes.json')

print("Downloads triggered. Check your browser's download bar.")